# Workout Exercise Classification — Final Report

A CNN-based classifier that identifies which of 22 gym exercises is being performed from a
sequence of frames. This notebook is the project write-up: what the data is, how it was
split, how the model was built and trained, and the final results.

**Best result: 84.7% accuracy** on the held-out evaluation set (efficientnet_b0 backbone,
fine-tuned end to end, with label smoothing + class-weighted loss + higher weight decay
as regularization). See [Results](#results) at the bottom for the full breakdown.

Code lives in `src/` (data pipeline, dataset, model, training loop) and `notebooks/`
(the actual Colab/local training and inference notebooks this report summarizes).

## 1. The data

[Workout/Exercise Images](https://www.kaggle.com/datasets/hasyimabdillah/workoutexercises-images)
from Kaggle — **22 exercise classes** (barbell biceps curl, bench press, squat, push up,
etc.), ~13,000 individual photos total. Despite the folder structure looking like video
frames, the source data is **standalone photos**, not real video — there's no camera/session
metadata, just filenames.

To still train a sequence model, filenames are parsed to group photos into pseudo-clips
(`data_splitter.parse_clip_frame_token`): a numeric suffix in the filename buckets photos
into groups of up to 10,000, which stand in for "frames of a clip". This gives **1,102
pseudo-clips** across the 22 classes, each padded/sampled down to a fixed **16 frames**.

## 2. How the data was split

Splitting is **stratified per class**: within each class, clips are further bucketed by
(frame-count, image-resolution) — `length_bucket` × `resolution_bucket` — shuffled within
bucket, then interleaved across buckets before the train/val/test cut. This keeps clips of
unusual length or resolution spread evenly across splits instead of clumping into one,
done independently per class so every class contributes proportionally to train/val/test
(`data_splitter.assign_split_frames`).

The final config uses `train_frac: 0.85`, `val_frac: 0.15`, `test_frac: 0.0` — no
separate held-out test split. **The validation set is used as the final evaluation set**
throughout this report (labeled "test" in results below, since that's its role here) —
with a dataset this small, a three-way split left too little data in each piece to be
reliable; two-way (train/val) makes better use of it.

## 3. Class imbalance

22 classes is a lot for ~1,100 clips (~50 per class on average) — and the classes aren't
balanced: the largest (*tricep pushdown*, 85 clips) has **6.1x** as many clips as the
smallest (*romanian deadlift*, 14 clips). Left unaddressed, a model can partly "solve"
training loss by fitting the common classes well and just memorizing the few examples of
rare ones, instead of learning to generalize on them. Addressed via **class-weighted
loss** (`training_utils.compute_class_weights`) — see [Training](#training).

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
for _candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (_candidate / 'configs').is_dir() and (_candidate / 'src').is_dir():
        PROJECT_ROOT = _candidate
        break

sequence_manifest = pd.read_csv(PROJECT_ROOT / 'artifacts' / 'sequence_manifest_len16.csv')
counts = sequence_manifest['class'].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(counts.index, counts.values, color='#4C72B0')
ax.set_ylabel('Clips')
ax.set_title(f'Clips per class ({counts.max()} max / {counts.min()} min = {counts.max()/counts.min():.1f}x)')
plt.xticks(rotation=90)
fig.tight_layout()
plt.show()

print(f'{len(sequence_manifest)} total clips across {sequence_manifest["class"].nunique()} classes')

## 4. Dataset & augmentation

Each clip is represented as **16 frames**, resized to 224×224 and ImageNet-normalized
(`dataset.load_image`). Images are pre-resized once into a cache (`ensure_image_cache`) so
repeated epochs don't re-decode the original files.

**Frame sampling.** For the *val* split, the 16 frames are chosen **once**, deterministically
(evenly spaced across the clip) — the same frames every epoch, for reproducible evaluation.
For the **train split**, frames are **re-sampled every epoch** (`WorkoutSequenceDataset`
with `augment=True`): the clip is divided into 16 segments and a *random* frame is drawn
from each segment on every read (`data_splitter.sample_random_indices`) — this is standard
temporal-segment-style jitter, giving the model a different view of each training clip
across epochs instead of memorizing one fixed 16-frame snapshot.

**Mirroring.** Each training clip is also randomly flipped horizontally (the whole clip
flipped consistently, not per-frame, to keep it visually coherent) — a coin flip per read,
applied after frame sampling.

## 5. Model — what we tried, what worked

Architecture: a per-frame CNN encoder → mean-pool across the 16 frames → linear
classifier head (`model.SequenceClassifier`). The encoder is swappable:

| Backbone | Setup | Result |
|---|---|---|
| Custom CNN (from scratch) | small conv net, frozen nothing | ~22% val_acc — not enough data to train a CNN from scratch |
| resnet18 (pretrained) | frozen backbone, embedding_dim 128/256 | 68.1% / 72.3% val_acc |
| efficientnet_b0 (pretrained) | frozen backbone | 73.5% val_acc — best of the frozen-backbone sweep |
| **efficientnet_b0 (pretrained)** | **fine-tuned end to end + regularized** | **84.7% val_acc — current best** |

The big jump came from **unfreezing the backbone** (fine-tuning all of efficientnet_b0,
not just a small head on top of frozen features) combined with regularization strong
enough to keep that larger trainable model from overfitting a dataset this small — see
below.

## 6. Training

**5-fold cross-validation.** With ~1,100 clips, a single fixed val split is noisy — one
lucky or unlucky split can make a config look better or worse than it is. A k-fold sweep
(`training_utils.run_kfold_sweep`, in the exploratory `04_train_kfold_colab.ipynb`)
rotated 5 train/val splits from one config, each sharing the same held-out slice, to get a
mean±std accuracy instead of trusting a single split. This validated that the frozen-backbone
approach was a real, stable ~73% — not a lucky split — and gave the confidence to move to a
bigger, riskier change (fine-tuning the whole backbone) for the next round.

**Regularizing the fine-tuned model.** Fine-tuning the full backbone gives the model far
more trainable parameters, which risks overfitting a ~950-clip training set badly. Three
regularizers were added on top of the fine-tuned setup:
- **`weight_decay: 0.001`** (10x the frozen-backbone baseline) — penalizes large weights.
- **`label_smoothing: 0.1`** — softens the training targets, discouraging the model from
  driving predictions to overconfident near-100% train accuracy.
- **Class-weighted loss** (`max_count / count` per class) — makes mistakes on
  under-represented classes cost more, directly countering the 6.1x imbalance from
  section 3.

**Schedule.** AdamW, `lr=0.001`, `StepLR` decaying by 10x every 5 epochs, up to 50 epochs
with early stopping (`patience=4`) on val accuracy.

## 7. Results {#results}

Evaluated on the **144-clip validation split** (see [section 2](#2.-How-the-data-was-split)
for why this doubles as the final evaluation set) — single evenly-spaced 16-frame window
per clip, no augmentation.

In [ ]:
predictions = pd.read_csv(PROJECT_ROOT / 'artifacts' / 'efficientnet_b0' / 'predictions.csv')
accuracy = predictions['correct'].mean()
print(f'Overall accuracy: {accuracy:.1%}  ({predictions["correct"].sum()} / {len(predictions)} clips)')

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from training_utils import plot_confusion_matrix

confusion_df = pd.read_csv(PROJECT_ROOT / 'artifacts' / 'efficientnet_b0' / 'confusion_matrix.csv', index_col=0)
plot_confusion_matrix(confusion_df, title='Validation set confusion matrix (efficientnet_b0, fine-tuned)')
plt.show()

In [ ]:
report_df = pd.read_csv(PROJECT_ROOT / 'artifacts' / 'efficientnet_b0' / 'classification_report.csv', index_col=0)
report_df.round(3)

**Reading the results.** Most classes score well (several classes hit 100% precision or
recall on their small val counts - `lateral raises`, `russian twist`, `squat`), and the
confusion matrix is strongly diagonal - most errors are between mechanically similar
exercises (e.g. `pull up` / `hip thrust` mix-ups, or `shoulder press` confusions), which is
a reasonable failure mode rather than random noise. `romanian deadlift`'s recall is
perfect but precision is only 50% - it's the smallest class (1 val clip, 14 train), so
that one number swings a lot on very little evidence; not something to over-read.

**A promising follow-up, not yet independently verified.** A later run with a larger
training split (`train_frac: 0.9` / `val_frac: 0.1`, otherwise the same recipe) logged
val_acc up to **89.9%** (plateauing around 87.6%) in its own training log. That number
comes from the training run itself, not from re-running inference against saved weights
like the result above - the checkpoint for that run hasn't been evaluated independently
yet. Worth doing before treating 90% as the real number.

## Reproducing this

- `notebooks/03_train_colab.ipynb` — training (Colab or local), config-driven via `configs/base.yaml`.
- `notebooks/05_inference.ipynb` — standalone checkpoint evaluation (what produced the results above).
- `notebooks/01_dataset_exploration.ipynb` — earlier data exploration.
- `src/` — the actual pipeline code (`data_splitter.py`, `dataset.py`, `model.py`, `pytorch_lightning.py`, `training_utils.py`).